# Exam exercises

Work on those you like the most. For the exam, preparing 2 of these is enough (or say 3 if you pick those that are very short).

## Q8: Shopping cart

Let's write a simple shopping cart class -- this will hold items that you intend to purchase as well as the amount, etc.  And allow you to add / remove items, get a subtotal, etc.

We'll use two classes: `Item` will be a single item and `ShoppingCart` will be the collection of items you wish to purchase.

First, our store needs an inventory -- here's what we have for sale:

In [1]:
INVENTORY_TEXT = """
apple, 0.60
banana, 0.20
grapefruit, 0.75
grapes, 1.99
kiwi, 0.50
lemon, 0.20
lime, 0.25
mango, 1.50
papaya, 2.95
pineapple, 3.50
blueberries, 1.99
blackberries, 2.50
peach, 0.50
plum, 0.33
clementine, 0.25
cantaloupe, 3.25
pear, 1.25
quince, 0.45
orange, 0.60
"""

# this will be a global -- convention is all caps
INVENTORY = {}
for line in INVENTORY_TEXT.splitlines():
    if line.strip() == "":
        continue
    item, price = line.split(",")
    INVENTORY[item] = float(price)


In [2]:
INVENTORY

{'apple': 0.6,
 'banana': 0.2,
 'grapefruit': 0.75,
 'grapes': 1.99,
 'kiwi': 0.5,
 'lemon': 0.2,
 'lime': 0.25,
 'mango': 1.5,
 'papaya': 2.95,
 'pineapple': 3.5,
 'blueberries': 1.99,
 'blackberries': 2.5,
 'peach': 0.5,
 'plum': 0.33,
 'clementine': 0.25,
 'cantaloupe': 3.25,
 'pear': 1.25,
 'quince': 0.45,
 'orange': 0.6}

### Item 

Let's write an item class now -- we want it to hold the name and quantity.  

You should have the following features:

* The name should be something in our inventory

* Our shopping cart will include a list of all the items we want to buy, so we want to be able to check for duplicates.  Implement the equal test, `==`, using `__eq__`

* We'll want to consolidate duplicates, so implement the `+` operator, using `__add__` so we can add items together in our shopping cart.  Note, add should raise a ValueError if you try to add two `Items` that don't have the same name.

Here's a start:

In [3]:
class Item:
    """ an item to buy """
    
    def __init__(self, name, quantity=1):
        """keep track of an item that is in our inventory"""
        if name not in INVENTORY:
            raise ValueError("invalid item name")
        self.name = name
        self.quantity = quantity
        
    def __repr__(self):
        return "{}: {}".format(self.name, self.quantity)
        
    def __eq__(self, other):
        """check if the items have the same name"""
        return self.name == other.name
    
    def __add__(self, other):
        """add two items together if they are the same type"""
        if self.name == other.name:
            return Item(self.name, self.quantity + other.quantity)
        else:
            raise ValueError("names don't match")

Here are some tests your code should pass:

In [4]:
a = Item("apple", 10)
b = Item("banana", 20)

In [5]:
INVENTORY[a.name]

0.6

In [6]:
c = Item("apple", 20)

In [7]:
# won't work
a + b

ValueError: names don't match

In [8]:
# will work
a += c
print(a)

apple: 30


In [9]:
d = Item("dog")

ValueError: invalid item name

In [10]:
# should be False
a == b

False

In [11]:
# should be True -- they have the same name
a == c

True

How do they behave in a list?

In [12]:
items = []
items.append(a)
items.append(b)
items

[apple: 30, banana: 20]

In [13]:
# should be True -- they have the same name
c in items

True

### ShoppingCart

Now we want to create a shopping cart.  The main thing it will do is hold a list of items.

In [14]:
class ShoppingCart:
    
    def __init__(self):
        # the list of items we control
        self.items = []
        
    def subtotal(self):
        """ return a subtotal of our items """
        total = 0.0
        for item in self.items:
            total += INVENTORY[item.name] * item.quantity
        return total

    def add(self, name, quantity):
        """ add an item to our cart -- if an item of the same name already
        exists, then increment the quantity.  Otherwise, add a new item
        to the cart with the desired quantity."""
        new_item = Item(name, quantity)
        if name in list(item.name for item in self.items):
            self.items[self.items.index(Item(name))] += new_item
        else:
            self.items.append(new_item)
        
    def remove(self, name):
        """ remove all of item name from the cart """
        try:
            self.items.remove(Item(name))
        except ValueError:
            print('No item of this kind in the cart.')
        
    def report(self):
        """ print a summary of the cart """
        for item in self.items:
            print(f"{item.name} : {item.quantity}")
        if not self.items:
            print('No items in cart.')

Here are some tests

In [15]:
sc = ShoppingCart()
sc.report()
sc.add("orange", 19)

No items in cart.


In [16]:
sc.add("apple", 2)

In [17]:
sc.report()

orange : 19
apple : 2


In [18]:
sc.add("apple", 9)

In [19]:
# apple should only be listed once in the report, with a quantity of 11
sc.report()

orange : 19
apple : 11


In [20]:
sc.subtotal()

18.0

In [21]:
sc.remove("apple")

In [22]:
# apple should no longer be listed
sc.report()

orange : 19


In [23]:
sc.remove("apple")

No item of this kind in the cart.


## Q9: Poker odds

Use the deck of cards class from the notebook we worked through class to write a _Monte Carlo_ code that plays a lot of hands of straight poker (like 100,000).  Count how many of these hands has a particular poker hand (like 3-of-a-kind).  The ratio of # of hands with 3-of-a-kind to total hands is an approximation to the odds of getting a 3-of-a-kind in poker.

### Bonus: 
Just to practice modules, write that into a `.py` file to allow you to import and reuse them here.

In [24]:
class Card:
    
    def __init__(self, suit=1, rank=2):
        if suit < 1 or suit > 4:
            print("invalid suit, setting to 1")
            suit = 1
            
        self.suit = suit
        self.rank = rank
        
    def value(self):
        """ we want things order primarily by rank then suit """
        return self.suit + (self.rank-1)*14
    
    # we include this to allow for comparisons with < and > between cards 
    def __lt__(self, other):
        return self.value() < other.value()

    def __eq__(self, other):
        return self.rank == other.rank and self.suit == other.suit
    
    def __repr__(self):
        return self.__str__()
    
    def __str__(self):
        suits = [u"\u2660",  # spade
                 u"\u2665",  # heart
                 u"\u2666",  # diamond
                 u"\u2663"]  # club
        
        r = str(self.rank)
        if self.rank == 11:
            r = "J"
        elif self.rank == 12:
            r = "Q"
        elif self.rank == 13:
            r = "K"
        elif self.rank == 14:
            r = "A"
                
        return r +':'+suits[self.suit-1]

In [25]:
import random

class Deck:
    """ the deck is a collection of cards """

    def __init__(self):

        self.nsuits = 4
        self.nranks = 13
        self.minrank = 2
        self.maxrank = self.minrank + self.nranks - 1

        self.cards = []

        for rank in range(self.minrank,self.maxrank+1):
            for suit in range(1, self.nsuits+1):
                self.cards.append(Card(rank=rank, suit=suit))

    def shuffle(self):
        random.shuffle(self.cards)

    def get_cards(self, num=1):
        hand = []
        for n in range(num):
            hand.append(self.cards.pop())

        return hand
    
    def __str__(self):
        string = ""
        for c in self.cards:
            string += str(c) + " "
        return string

    def reset(self):
        '''put all cards back in the deck'''
        self.__init__()

In [26]:
def pokerhand(hand):
    '''check if the hand is a three of a kind, 
    meaning three cards of one rank and two of
    different ones.'''
    ranks = list(card.rank for card in hand)
    for i in range(2, 15):
        if ranks.count(i) == 3 and len(set(ranks)) == 3:
                return True
    return False

In [27]:
nhands = 1e5
threeofakind = 0

mydeck = Deck()
for i in range(int(nhands)):
    #shuffle deck, get the cards and check hand
    mydeck.shuffle()
    hand = mydeck.get_cards(num=5)
    if pokerhand(hand): threeofakind+=1
        
    #put the hand back in
    mydeck.reset()

print('Three-of-a-kind to hands ratio: {}'.format(threeofakind/nhands))

Three-of-a-kind to hands ratio: 0.02068


#### Doing the same but w/ a module:

In [28]:
import poker as pk

In [29]:
newdeck = pk.Deck()
print(newdeck)

2:♠ 2:♥ 2:♦ 2:♣ 3:♠ 3:♥ 3:♦ 3:♣ 4:♠ 4:♥ 4:♦ 4:♣ 5:♠ 5:♥ 5:♦ 5:♣ 6:♠ 6:♥ 6:♦ 6:♣ 7:♠ 7:♥ 7:♦ 7:♣ 8:♠ 8:♥ 8:♦ 8:♣ 9:♠ 9:♥ 9:♦ 9:♣ 10:♠ 10:♥ 10:♦ 10:♣ J:♠ J:♥ J:♦ J:♣ Q:♠ Q:♥ Q:♦ Q:♣ K:♠ K:♥ K:♦ K:♣ A:♠ A:♥ A:♦ A:♣ 


In [30]:
nhands = 1e5
threeofakind = 0

for i in range(int(nhands)):
    #shuffle deck, get the cards and check hand
    newdeck.shuffle()
    hand = newdeck.get_cards(num=5)
    if pk.pokerhand(hand): threeofakind+=1
        
    #put the hand back in
    newdeck.reset()

print('Three-of-a-kind to hands ratio: {}'.format(threeofakind/nhands))

Three-of-a-kind to hands ratio: 0.02125
